# NeuroObfuscator v7.1 — QLoRA fine-tuning (A100/L4 GPU, 7B)

Пресет **захардкожен**: Qwen2.5-Coder-7B-Instruct 4-bit, batch 16 x 1 (на L4 верните 8 x 2 в ячейке пресета). Автодетекта нет.
Обучение **conditional** модели: промпт содержит `Target intensity: light|medium|heavy`, и модель обязана ему следовать.

**Датасет v7**: `data/final_v7/{train,val}.jsonl` — 6 000 train / 750 val (загрузить архив `final_v7.zip` в Google Drive).

**Что нового в v7 (почему переобучение обязательно):**
- Fixed mode collapse: top order 60.5% -> **14.0%**, 26 уникальных порядков (было 24, но 60% массы на одном)
- Intensity баланс: 28.4 / 47.5 / 24.2% (light/medium/heavy) — модель перестанет выдавать heavy в 47% случаев
- string_encode покрытие 99.9% (было ~30%) — топ-порядок v6 вообще не содержал string_encode
- 0 противоречий R1–R6 (в v6: 1234)
- 2 737 функций имеют 2–3 intensity-варианта в train (контраст учит модель слушаться `Target intensity`)
- Промпт-формат изменился (строка `Target intensity: X`) — чекпоинт v6 несовместим

**DPO**: после SFT можно дообучить на `data/final_v7/dpo_pairs_v7.jsonl` (19 937 пар, conditional) — отдельная стадия.

## Железо и пресеты

| Пресет | GPU | Модель | Batch |
|---|---|---|---|
| `t4_3b` (основной на T4) | T4 15 GB | Qwen2.5-Coder-3B-Instruct, 4-bit | 8 x 2 = 16 |
| `l4_qwen7b` (рекомендуемый на L4) | L4 22.5 GB | Qwen2.5-Coder-7B-Instruct, 4-bit | 8 x 2 = 16 |
| `l4_codellama7b` | L4 22.5 GB | CodeLlama-7B-Instruct, 4-bit | 8 x 2 = 16 |

Пресет выбирается **автоматически** по VRAM (порог 20 GB), можно задать вручную (`MANUAL_PRESET`).

Гиперпараметры: r=32, alpha=64, lr=2e-4, 3 эпохи, cosine, effective batch 16.

**Время на T4 (3B)**: ~40-70 мин на 3 эпохи.


In [ ]:
%pip install unsloth
# Если Colab предложит перезапустить runtime — перезапустить и продолжить отсюда


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/neuroobfuscator/final_v7'  # поправить под свой путь в Drive
ADAPTER_OUT = '/content/drive/MyDrive/neuroobfuscator/adapters_v7'
!ls {DATA_DIR}


In [ ]:
import torch

# v7.1: пресет захардкожен на 7B (A100/L4). Автодетект по VRAM убран.
PRESET_NAME = 'l4_qwen7b'

PRESETS = {
    'l4_qwen7b': {
        'model_name': 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit',
        'max_seq_len': 2048,
        'batch_size': 16,   # A100 40GB: 16 x 1 (эффективный батч 16); на L4 верните 8 x 2
        'grad_accum': 1,
    },
}

CFG = PRESETS[PRESET_NAME]
MODEL_NAME = CFG['model_name']
MAX_SEQ_LEN = CFG['max_seq_len']
IS_QWEN = 'qwen' in MODEL_NAME.lower()
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB) -> preset {PRESET_NAME}')
print('model:', MODEL_NAME, '| bs:', CFG['batch_size'], 'x ga:', CFG['grad_accum'], '| qwen:', IS_QWEN)


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG['model_name'],
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('LoRA ready')


In [ ]:
# Датасет: keys = {id, instruction, output, metadata}
# instruction = [INST]-промпт, содержит 'Target intensity: X' (conditional v7)
# output = чистый JSON без seed; id = f'{fid}@{intensity}'
import json, re
from collections import Counter
from datasets import Dataset

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return Dataset.from_list([json.loads(l) for l in f if l.strip()])

train_ds = load_jsonl(f'{DATA_DIR}/train.jsonl')
val_ds = load_jsonl(f'{DATA_DIR}/val.jsonl')
print('train:', len(train_ds), '| val:', len(val_ds))

# Санити-проверка conditional-формата: все промпты должны нести Target intensity
def target_intensity_of(instruction):
    m = re.search(r'Target intensity: (light|medium|heavy)', instruction)
    return m.group(1) if m else None

missing = sum(1 for r in train_ds if target_intensity_of(r['instruction']) is None)
assert missing == 0, f'{missing} train records without Target intensity line — это не v7 датасет!'
intensities = Counter(target_intensity_of(r['instruction']) for r in train_ds)
print('train intensity targets:', dict(intensities))

EOS = tokenizer.eos_token

def format_example(rec):
    return {'text': rec['instruction'] + rec['output'] + EOS}

train_fmt = train_ds.map(format_example)
val_fmt = val_ds.map(format_example)
print(repr(train_fmt[0]['text'][-160:]))


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_fmt,
    eval_dataset=val_fmt,
    args=SFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=CFG['batch_size'],
        gradient_accumulation_steps=CFG['grad_accum'],
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=100,
        save_strategy='steps',
        save_steps=200,
        save_total_limit=2,
        per_device_eval_batch_size=16,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit',
        weight_decay=0.01,
        seed=3407,
        output_dir='/content/checkpoints',
        report_to='none',
        max_seq_length=MAX_SEQ_LEN,
    ),
)


In [ ]:
# Маскирование промпта: loss только на JSON-комплишене.
# ВАЖНО: маркер всегда '[/INST]' — наш raw-промпт кормится модели без ChatML-шаблона,
# поэтому ассистент-маркера Qwen ('<|im_start|>assistant') в тексте НЕТ: его поиск
# даёт idx=-1 и маскирует ВСЕ токены в -100 (AssertionError). Так же, как в v5.
from transformers import DataCollatorForSeq2Seq

RESPONSE_MARKER = '[/INST]'

def tokenize_with_mask(rec):
    full = rec['text']
    enc = tokenizer(
        full,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_offsets_mapping=True,
    )
    idx = full.rfind(RESPONSE_MARKER)
    assert idx != -1, 'prompt does not contain [/INST] — не тот формат инструкции'
    resp_start = idx + len(RESPONSE_MARKER)
    input_ids = enc['input_ids']
    labels = [
        (tid if b > resp_start else -100)
        for tid, (a, b) in zip(input_ids, enc['offset_mapping'])
    ]
    return {'input_ids': input_ids, 'labels': labels}

train_tok = train_fmt.map(
    tokenize_with_mask,
    remove_columns=train_fmt.column_names,
    desc='Tokenize + mask prompt',
)
val_tok = val_fmt.map(
    tokenize_with_mask,
    remove_columns=val_fmt.column_names,
    desc='Tokenize + mask prompt',
)

assert all(any(l != -100 for l in ex['labels']) for ex in train_tok.select(range(64))), \
    'все labels = -100: маркер [/INST] не найден'

# Диагностика: доля обучающих токенов и хвост первого примера (должен быть JSON)
unmasked = masked = 0
for ex in train_tok.select(range(min(256, len(train_tok)))):
    unmasked += sum(1 for l in ex['labels'] if l != -100)
    masked += len(ex['labels'])
print(f'tokens: {masked}, loss tokens: {unmasked} ({unmasked/masked:.1%})')
example = train_tok[0]
print('loss tail of train_tok[0]:',
      repr(tokenizer.decode([t for t, l in zip(example['input_ids'], example['labels']) if l != -100][:80])[:200]))

trainer.train_dataset = train_tok
trainer.eval_dataset = val_tok.select(range(min(200, len(val_tok))))
trainer.data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)
print('prompt-masked datasets ready:', len(train_tok), 'train,', len(trainer.eval_dataset), 'eval-subset')


In [ ]:
trainer_stats = trainer.train()
print(f'train runtime: {trainer_stats.metrics.get("train_runtime", 0)/60:.1f} min')
print(f'train loss:    {trainer_stats.metrics.get("train_loss", 0):.4f}')


In [ ]:
# Быстрая проверка: генерация на валидационном примере (greedy)
FastLanguageModel.for_inference(model)

sample = val_ds[0]
inputs = tokenizer(sample['instruction'], return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=False)
generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('TARGET INTENSITY:', target_intensity_of(sample['instruction']))
print('EXPECTED:', sample['output'])
print('GENERATED:', generated)


In [ ]:
# Метрики v7.1: JSON parse + schema + intensity obedience + light purity
import json

def extract_json(s):
    a, b = s.find('{'), s.rfind('}')
    if a == -1 or b <= a:
        return None
    try:
        return json.loads(s[a:b + 1])
    except Exception:
        return None

ORDER = ["rename", "string_encode", "operator_sub", "dead_code", "opaque_predicates"]

def validate_plan_schema(plan):
    if not isinstance(plan, dict): return False
    if not all(k in plan for k in ['intensity', 'transforms', 'order']): return False
    if plan['intensity'] not in {'light', 'medium', 'heavy'}: return False
    if set(plan.get('transforms', {})) != set(ORDER): return False
    enabled = [n for n in ORDER if plan['transforms'][n].get('enabled')]
    return plan.get('order') == enabled

def shape_matches(plan, tgt):
    if tgt is None: return True
    order = set(plan.get('order', []))
    if tgt == 'light':  return order <= {'rename', 'dead_code'}
    if tgt == 'medium': return 'opaque_predicates' not in order and bool(order - {'rename', 'dead_code'})
    if tgt == 'heavy':  return 'opaque_predicates' in order
    return True

def eval_batch(N):
    ok = schema_ok = field_obey = shape_ok = 0
    light_total = light_pure = 0
    orders = Counter(); non_light_orders = Counter()
    failures = []
    for i in range(min(N, len(val_ds))):
        rec = val_ds[i]
        tgt = target_intensity_of(rec['instruction'])
        inputs = tokenizer(rec['instruction'], return_tensors='pt').to('cuda')
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        plan = extract_json(gen)
        if plan is None:
            failures.append(('json', i, gen[:120])); continue
        ok += 1
        if not validate_plan_schema(plan):
            failures.append(('schema', i, json.dumps(plan)[:120])); continue
        schema_ok += 1
        orders[','.join(plan.get('order', []))] += 1
        if tgt is not None and plan.get('intensity') == tgt:
            field_obey += 1
        if shape_matches(plan, tgt):
            shape_ok += 1
        if tgt == 'light':
            light_total += 1
            if set(plan.get('order', [])) <= {'rename', 'dead_code'}:
                light_pure += 1
        else:
            non_light_orders[','.join(plan.get('order', []))] += 1
    return {'total': ok, 'schema_ok': schema_ok, 'field_obey': field_obey, 'shape_ok': shape_ok,
            'light_pure': light_pure, 'light_total': light_total,
            'orders': orders, 'non_light_orders': non_light_orders, 'failures': failures}

r = eval_batch(64)
n = r['total']
nl = sum(r['non_light_orders'].values())
top_nl, top_nl_c = (r['non_light_orders'].most_common(1)[0] if r['non_light_orders'] else ('none', 0))
print(f'JSON parse rate:       {n}/64 = {n/64:.1%} (target >= 95%)')
print(f'Schema valid rate:     {r["schema_ok"]}/64 = {r["schema_ok"]/64:.1%} (target >= 90%)')
print(f'Field obedience:       {r["field_obey"]}/64 = {r["field_obey"]/64:.1%} (target >= 90%)')
print(f'SHAPE obedience:       {r["shape_ok"]}/64 = {r["shape_ok"]/64:.1%} (target >= 90%)')
print(f'Light purity:          {r["light_pure"]}/{r["light_total"]} = {r["light_pure"]/max(r["light_total"],1):.1%} (target >= 95%)')
print(f'Unique orders:         {len(r["orders"])} (info; внутриклассовое разнообразие)')
print(f'Top NON-LIGHT order:   {top_nl.replace(",", " > ")} {top_nl_c/max(nl,1):.1%} (target <= 45%)')
for kind, i, snippet in r['failures'][:5]:
    print(f'  [{kind}] val_ds[{i}]: {snippet}')


In [ ]:
# Финальная оценка на 200 val-примерах + DIVERSITY-метрики v7.1
# Гейты: JSON >= 95%, schema >= 90%, obedience >= 90% (поле+контент),
#        light purity >= 95%, top non-light order <= 45%.
# ВАЖНО: light намеренно даёт один порядок (доля = light-квоте ~20-23%),
# поэтому diversity меряем по non-light порядкам (см. v7.1 в plan.md).
import os
os.makedirs(ADAPTER_OUT, exist_ok=True)
r = eval_batch(200)
n = r['total']
nl = sum(r['non_light_orders'].values())
top_nl, top_nl_c = (r['non_light_orders'].most_common(1)[0] if r['non_light_orders'] else ('none', 0))

print(f'JSON parse rate:       {n}/200 = {n/200:.1%} (target >= 95%)')
print(f'Schema valid rate:     {r["schema_ok"]}/200 = {r["schema_ok"]/200:.1%} (target >= 90%)')
print(f'Field obedience:       {r["field_obey"]}/200 = {r["field_obey"]/200:.1%} (target >= 90%)')
print(f'SHAPE obedience:       {r["shape_ok"]}/200 = {r["shape_ok"]/200:.1%} (target >= 90%)')
print(f'Light purity:          {r["light_pure"]}/{r["light_total"]} = {r["light_pure"]/max(r["light_total"],1):.1%} (target >= 95%)')
print(f'Unique orders:         {len(r["orders"])} | non-light: {len(r["non_light_orders"])}')
print(f'Top NON-LIGHT order:   {top_nl.replace(",", " > ")} {top_nl_c/max(nl,1):.1%} (target <= 45%)')
print()
print('Top-5 generated orders:')
for o, c in r['orders'].most_common(5):
    tag = ' [light]' if set(o.split(',')) <= {'rename', 'dead_code'} else ''
    print(f'  {c:4d}  {c/200:5.1%}  {o.replace(",", " > ")}{tag}')
if r['failures']:
    print(f'\nFailures ({len(r["failures"])}):')
    for kind, i, snippet in r['failures'][:5]:
        print(f'  [{kind}] val_ds[{i}]: {snippet}')

report = {
    'model': MODEL_NAME,
    'dataset': 'final_v7.1 (conditional shapes)',
    'n_eval': 200,
    'json_rate': n / 200,
    'schema_rate': r['schema_ok'] / 200,
    'field_obedience_rate': r['field_obey'] / 200,
    'shape_obedience_rate': r['shape_ok'] / 200,
    'light_purity': r['light_pure'] / max(r['light_total'], 1),
    'unique_orders': len(r['orders']),
    'unique_non_light_orders': len(r['non_light_orders']),
    'top_non_light_order': top_nl,
    'top_non_light_order_share': top_nl_c / max(nl, 1),
}
with open(f'{ADAPTER_OUT}/train_v7_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('\nreport ->', f'{ADAPTER_OUT}/train_v7_report.json')


In [ ]:
# Сохранить LoRA-адаптер на Drive (для inference.py / DPO-стадии)
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print('saved to', ADAPTER_OUT)


In [ ]:
# Экспорт в GGUF (для llama.cpp / Ollama / LM Studio)
# q4_k_m: ~5 GB для 7B, ~2 GB для 3B — лучший баланс размер/качество
# альтернатива: 'q8_0' (~8.5 GB для 7B, почти без потерь качества)

GGUF_OUT = '/content/drive/MyDrive/neuroobfuscator/gguf_v7'

model.save_pretrained_gguf(
    GGUF_OUT,
    tokenizer,
    quantization_method='q4_k_m',
)
print('GGUF saved to', GGUF_OUT)
!ls -la {GGUF_OUT}


## Следующие шаги после SFT

1. **DPO-стадия** (рекомендуется): дообучить адаптер на `data/final_v7/dpo_pairs_v7.jsonl` (19 937 conditional-пар, gap >= 0.05) — дополнительно давит mode collapse.
2. **Eval v7** (`scripts/evaluate.py` + inference-ноутбук): прогнать test 750 + проверка obedience/diversity; сравнить с v6 (semantic 100%, но 8 orders / 62.3% top).
3. Обновить `engine_bundle` не нужно (движок v6.1 не менялся) — на Drive уже актуальный.
4. Gradio demo: добавить селектор intensity (light/medium/heavy) → `inference.py --intensity`.
